# Ray Cluster Connection Template

This notebook connects to your cloud Ray cluster for distributed computing.

In [1]:
import ray
import pyarrow.fs
from ray import tune
from ray.train import RunConfig

## 1. Port Forward Ray Cluster

In a terminal, run:
```bash
kubectl port-forward -n ray-system svc/raycluster-sample-head-svc 10001:10001 8265:8265
```

In [2]:
# Connect to Ray cluster
ray.init("ray://localhost:10001")

# Verify connection
print(f"Connected to Ray cluster!")
print(f"Available resources: {ray.available_resources()}")

2025-10-26 12:36:04,290	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver
SIGTERM handler is not set because current thread is not the main thread.


Connected to Ray cluster!
Available resources: {'object_store_memory': 6256174693.0, 'node:10.244.117.199': 1.0, 'node:10.244.211.7': 1.0, 'node:10.244.19.135': 1.0, 'memory': 22548578304.0, 'CPU': 10.0, 'node:__internal_head__': 1.0, 'node:10.244.122.7': 1.0, 'node:10.244.4.204': 1.0}


## 2. Setup MinIO Storage (for results)

In [3]:
# Setup MinIO S3 filesystem
s3_fs = pyarrow.fs.S3FileSystem(
    endpoint_override="localhost:9000",  # Port forward: kubectl port-forward -n minio svc/minio 9000:9000
    scheme="http",
    access_key="minioadmin",
    secret_key="minioadmin123",
    allow_bucket_creation=True
)

## 3. Test Remote Function

In [4]:
@ray.remote
def hello_ray():
    import socket
    return f"Hello from {socket.gethostname()}!"

# Run on cluster
result = ray.get(hello_ray.remote())
print(result)

Hello from raycluster-sample-head-vctz7!


## 4. Your Training Code Here

In [5]:
# Define your training function
def training_function(config):
    # Your code here
    pass

# Run Ray Tune
# tuner = tune.Tuner(
#     training_function,
#     param_space={...},
#     run_config=RunConfig(
#         storage_path="ray-results",
#         storage_filesystem=s3_fs
#     )
# )
# results = tuner.fit()

In [6]:
# Don't forget to shutdown when done
ray.shutdown()